# Notebook 6: Feature Importance ve Explainability
SHAP yerine manuel feature importance kullaniyoruz.

In [1]:
import sys
sys.path.append('..')
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from src.utils import load_model, print_section, FIGURES_DIR
print('Import basarili')

Import basarili


In [2]:
X_test  = np.load('../data/processed/X_test.npy')
y_test  = np.load('../data/processed/y_test.npy')
X_adv_combined = np.load('../data/processed/X_adv_combined.npy')
with open('../data/processed/feature_names.json') as f:
    feature_names = json.load(f)
rf_model  = load_model('baseline_random_forest')
xgb_model = load_model('baseline_xgboost')
print(f'Test: {X_test.shape} | Features: {len(feature_names)}')

[2026-02-28 12:38:50] INFO [utils] Model yüklendi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\data\models\baseline_random_forest.pkl
[2026-02-28 12:38:50] INFO [utils] Model yüklendi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\data\models\baseline_xgboost.pkl


Test: (25013, 64) | Features: 64


In [3]:
print_section('Random Forest Feature Importance')
rf_importances = rf_model.feature_importances_
top_idx = np.argsort(rf_importances)[::-1][:20]
top_names = [feature_names[i] for i in top_idx]
top_vals  = rf_importances[top_idx]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top_names[::-1], top_vals[::-1], color='#2196F3', alpha=0.8)
ax.set_title('Random Forest - Top 20 Feature Importance', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rf_feature_importance.png', dpi=150, bbox_inches='tight')
print('Grafik kaydedildi')
print('\nTop 10 Feature:')
for i, (name, val) in enumerate(zip(top_names[:10], top_vals[:10]), 1):
    print(f'  {i:2d}. {name}: {val:.4f}')


════════════════════════════════════════════════════════════
  Random Forest Feature Importance
════════════════════════════════════════════════════════════

Grafik kaydedildi

Top 10 Feature:
   1. is_https: 0.1720
   2. tld_trusted: 0.0858
   3. num_slashes: 0.0635
   4. tld_length: 0.0501
   5. subdomain_length: 0.0443
   6. path_length: 0.0349
   7. num_hyphens: 0.0345
   8. path_entropy: 0.0344
   9. special_ratio: 0.0343
  10. num_subdomains: 0.0325


In [4]:
print_section('XGBoost Feature Importance')
xgb_importances = xgb_model.feature_importances_
top_idx_xgb = np.argsort(xgb_importances)[::-1][:20]
top_names_xgb = [feature_names[i] for i in top_idx_xgb]
top_vals_xgb  = xgb_importances[top_idx_xgb]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top_names_xgb[::-1], top_vals_xgb[::-1], color='#FF9800', alpha=0.8)
ax.set_title('XGBoost - Top 20 Feature Importance', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'xgb_feature_importance.png', dpi=150, bbox_inches='tight')
print('Grafik kaydedildi')


════════════════════════════════════════════════════════════
  XGBoost Feature Importance
════════════════════════════════════════════════════════════

Grafik kaydedildi


In [5]:
print_section('Adversarial Oncesi vs Sonrasi Feature Degisimleri')
phish_idx = np.where(y_test == 1)[0][:200]
X_phish_normal = X_test[phish_idx]
X_phish_adv    = X_adv_combined[phish_idx]

diff = np.abs(X_phish_normal - X_phish_adv).mean(axis=0)
top_diff_idx = np.argsort(diff)[::-1][:15]
top_diff_names = [feature_names[i] for i in top_diff_idx]
top_diff_vals  = diff[top_diff_idx]

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top_diff_names[::-1], top_diff_vals[::-1], color='#F44336', alpha=0.8)
ax.set_title('Adversarial Saldiri: En Cok Degisen Featurelar', fontsize=13)
ax.set_xlabel('Ortalama Degisim Miktari')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'adversarial_feature_changes.png', dpi=150, bbox_inches='tight')
print('Grafik kaydedildi')
print('\nEn cok degisen featurelar:')
for name, val in zip(top_diff_names[:10], top_diff_vals[:10]):
    print(f'  {name}: {val:.4f}')


════════════════════════════════════════════════════════════
  Adversarial Oncesi vs Sonrasi Feature Degisimleri
════════════════════════════════════════════════════════════

Grafik kaydedildi

En cok degisen featurelar:
  url_length: 5.5077
  port: 1.6237
  domain_length: 1.3340
  min_brand_levenshtein: 0.6901
  is_https: 0.3507
  homoglyph_count: 0.2678
  num_hyphens: 0.1785
  num_subdomains: 0.1520
  has_homoglyph: 0.1457
  suspicious_keywords: 0.1410


In [6]:
print_section('Neden Phishing? - Ornek Aciklama')
predictions = rf_model.predict(X_test)
correct_phish = np.where((predictions == 1) & (y_test == 1))[0]

if len(correct_phish) > 0:
    idx = correct_phish[0]
    example = X_test[idx]
    prob = rf_model.predict_proba(example.reshape(1,-1))[0][1]
    
    print(f'Phishing olasiligi: {prob:.2%}')
    print('\nBu URL Phishing Cunku:')
    print('-'*50)
    
    top_features_idx = np.argsort(rf_importances)[::-1][:5]
    for rank, feat_i in enumerate(top_features_idx, 1):
        feat_name  = feature_names[feat_i]
        feat_value = example[feat_i]
        importance = rf_importances[feat_i]
        print(f'  {rank}. {feat_name}: {feat_value:.3f} (importance: {importance:.4f})')

print('\nNotebook 6 tamamlandi!')
print('Sonraki adim: 7_evaluation_report.ipynb')


════════════════════════════════════════════════════════════
  Neden Phishing? - Ornek Aciklama
════════════════════════════════════════════════════════════

Phishing olasiligi: 95.09%

Bu URL Phishing Cunku:
--------------------------------------------------
  1. is_https: 0.000 (importance: 0.1720)
  2. tld_trusted: 0.000 (importance: 0.0858)
  3. num_slashes: 5.000 (importance: 0.0635)
  4. tld_length: 6.000 (importance: 0.0501)
  5. subdomain_length: 3.000 (importance: 0.0443)

Notebook 6 tamamlandi!
Sonraki adim: 7_evaluation_report.ipynb
